# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Which city departments receive the most complaints?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-25 13:16:05 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-25T13:16:05.847385")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'city department complaints':

1. **NYPD Compstat Data 2022** (25.4% match)
   Resource: Volume 29 Week 06
   Resource ID: `9f0128d6-7a1c-4c39-91e5-e23cf6a68abb`
   Dataset ID: `deea1818-aa6d-4c00-9ea6-a70b8cae7572`
   Format: CSV | Rows: 1,445 | Cols: 20
   Tags: NYPD, NYC Crime, CompStat, Crime Statistics, 2022 Data, Weekly Crime, Felony Assault, Burglary, Robbery, Hate Crimes
   This dataset provides a weekly snapshot of New York City crime statistics for 2022, specifically for Volume 29, Week 06. It details the 'Current' number of reported crime
```


In [ ]:
# Step 1: Semantic Search Resources

# Semantic search via Pinecone vector store
# (requires PineconeVectorStore from data_concierge)
from data_concierge.data_layer.connectors.pinecone_store import PineconeVectorStore

store = PineconeVectorStore()
results = store.search_resources('city department complaints', n_results=10)
for r in results:
    print(f"{r['dataset_title']} — {r['resource_id']} (score: {r['score']:.2f})")


## Step 2: Semantic Search Resources


**Result preview:**
```
Found 10 semantically matching resources for 'Pittsburgh 311 service requests complaints departments':

1. **NYPD Compstat Data 2024** (23.6% match)
   Resource: Volume 31 Week 46
   Resource ID: `ad2c709a-5d85-4d3e-8104-0437b7547f5b`
   Dataset ID: `bf8d2bff-71c1-4fa6-a3a3-eb0c5c0a95de`
   Format: CSV | Rows: 1,446 | Cols: 20
   Tags: NYPD, CompStat, Crime Statistics, New York City, Criminal Incidents, Weekly Crime Data, Year-to-Date Crime, Crime Trends, Law Enforcement, Public Safety
   This dataset contains weekly CompStat data from the NYPD for Volume 31, Week 46 of 2024. It provides crime
```


In [ ]:
# Step 2: Semantic Search Resources

# Semantic search via Pinecone vector store
# (requires PineconeVectorStore from data_concierge)
from data_concierge.data_layer.connectors.pinecone_store import PineconeVectorStore

store = PineconeVectorStore()
results = store.search_resources('Pittsburgh 311 service requests complaints departments', n_results=10)
for r in results:
    print(f"{r['dataset_title']} — {r['resource_id']} (score: {r['score']:.2f})")


## Step 3: Search for Datasets

**Search query:** `311 complaints service requests`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 2 datasets matching '311 complaints service requests'

1. **Pittsburgh 311 Data**
   ID: `pittsburgh-311-data`
   City of Pittsburgh has transitioned to a new 311 system February 4, 2025 - this is the data from the new system.  This data set shows 311 service requests in the City of Pittsbur
   - 311 Data (CSV) [DataStore] ID: `5202679a-d243-402e-b82a-63189995a942`
   - 311 Data User Guide (HTML) ID: `726a22f9-7741-4fd8-be96-9c8574f41d50`
   - 311 Issue, Subject Code, and Public Access Level Codebook (XLSX) [DataStore] ID: `2de7cb02-b60f-4a54-99b4-bd633b0fa365`
   - 311 Data Dictionary
```


In [ ]:
# Step 3: Search for Datasets

# Search for datasets
import requests, json

params = {"q": '311 complaints service requests', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 4: Load Data from Resource

**Resource ID:** `5202679a-d243-402e-b82a-63189995a942`
**Limit:** 5

**Result preview:**
```
Resource: 5202679a-d243-402e-b82a-63189995a942
Total records: 938,363
Loaded: 5
Fields (25): case_number, status, case_owner, subject, subject_code, legacy_subject, created_date_et, created_date_utc, last_modified_date_et, last_modified_date_utc, closed_date_et, closed_date_utc, origin, street, street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy, unique_id

Sample (5 rows):

case_number status     case_owner     subject subject_code                   legacy_subject     created_date_et    created_date_utc last_modified_date_et last
```


In [ ]:
# Step 4: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5202679a-d243-402e-b82a-63189995a942', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 5: Load Data from Resource

**Resource ID:** `29462525-62a6-45bf-9b5e-ad2e1c06348d`
**Limit:** 5

**Result preview:**
```
Resource: 29462525-62a6-45bf-9b5e-ad2e1c06348d
Total records: 815,417
Loaded: 5
Fields (28): group_id, num_requests, parent_closed, status_name, status_code, dept, request_type_name, request_type_id, create_date_et, create_date_utc, last_action_et, last_action_utc, closed_date_et, closed_date_utc, origin, street, cross_street, street_id, cross_street_id, city, neighborhood, census_tract, council_district, ward, police_zone, latitude, longitude, geo_accuracy

Sample (5 rows):

group_id  num_requests  parent_closed status_name  status_code                 dept request_type_name request_type_id  
```


In [ ]:
# Step 5: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '29462525-62a6-45bf-9b5e-ad2e1c06348d', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT "dept", COUNT(*) as complaint_count FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d" WHERE "dept" IS NOT NULL AND "dept" != '' GROUP BY "dept" ORDER BY complaint_count DESC LIMIT 20
```

**Result preview:**
```
SQL: SELECT "dept", COUNT(*) as complaint_count FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d" WHERE "dept" IS NOT NULL AND "dept" != '' GROUP BY "dept" ORDER BY complaint_count DESC LIMIT 20
Rows: 20
Columns: dept, complaint_count

                                dept  complaint_count
   Permits, Licenses and Inspections           169802
            DPW - Street Maintenance           167941
                        DPW - Refuse           112828
                  Police - Zones 1-6            55941
                      DOMI - Permits            34221
Pittsburgh Water and Sewer Authority          
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "dept", COUNT(*) as complaint_count FROM "29462525-62a6-45bf-9b5e-ad2e1c06348d" WHERE "dept" IS NOT NULL AND "dept" != \'\' GROUP BY "dept" ORDER BY complaint_count DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 7: SQL Analysis Query

**SQL:**
```sql
SELECT "case_owner", COUNT(*) as complaint_count FROM "5202679a-d243-402e-b82a-63189995a942" WHERE "case_owner" IS NOT NULL AND "case_owner" != '' GROUP BY "case_owner" ORDER BY complaint_count DESC LIMIT 20
```

**Result preview:**
```
SQL: SELECT "case_owner", COUNT(*) as complaint_count FROM "5202679a-d243-402e-b82a-63189995a942" WHERE "case_owner" IS NOT NULL AND "case_owner" != '' GROUP BY "case_owner" ORDER BY complaint_count DESC LIMIT 20
Rows: 20
Columns: case_owner, complaint_count

                          case_owner  complaint_count
            DPW - Street Maintenance           202900
   Permits, Licenses and Inspections           167525
                        DPW - Refuse           112410
                  Police - Zones 1-6            51860
                      DOMI - Permits            41146
Pittsburgh Water
```


In [ ]:
# Step 7: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "case_owner", COUNT(*) as complaint_count FROM "5202679a-d243-402e-b82a-63189995a942" WHERE "case_owner" IS NOT NULL AND "case_owner" != \'\' GROUP BY "case_owner" ORDER BY complaint_count DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Excellent! I now have comprehensive data from both datasets. Here's a full breakdown:

---

## 🏙️ Pittsburgh 311 Complaints by City Department

Using data from the **Pittsburgh 311 Data** portal (WPRDC), which covers **over 1.75 million total service requests** across two datasets (archive through Feb 2025 + the new system), here are the top departments by complaint volume:

### 📊 Top Departments — All-Time Rankings

| Rank | Department | Archive Count | New System Count |
|------|-----------|--------------|-----------------|
| 🥇 1 | **DPW – Street Maintenance** | 167,941 | 202,900 |
| 🥈 2 | **Permits, Licenses & Inspections (PLI)** | 169,802 | 167,525 |
| 🥉 3 | **DPW – Refuse** | 112,828 | 112,410 |
| 4 | **Police – Zones 1–6** | 55,941 | 51,860 |
| 5 | **DOMI – Permits** | 34,221 | 41,146 |
| 6 | **Pittsburgh Water & Sewer Authority (PWSA)** | 30,236 | 29,302 |
| 7 | **DOMI – TrafficShop** | 27,242 | 26,512 |
| 8 | **Police – AVU** | 24,530 | 24,927 |
| 9 | **Animal Care & Control** | 23,255 | 23,233 |
| 10 | **DPW – Forestry Division** | 22,577 | 25,519 |

---

### 🔑 Key Takeaways

1. **DPW – Street Maintenance** and **Permits, Licenses & Inspections (PLI)** consistently battle for the #1 spot — each receiving well over **160,000+ complaints** historically. Street maintenance complaints (potholes, road damage, etc.) are the single largest category in the newer system.

2. **DPW – Refuse** (trash/recycling) ranks a solid #3 with over **112,000 complaints** in each system.

3. **Police** collectively (Zones 1–6 + AVU) account for **~80,000+** complaints, largely for parking violations and other non-emergency issues.

4. **DOMI (Department of Mobility & Infrastructure)** handles several large complaint areas — permits, traffic, asphalt — totaling over **80,000 requests** combined.

> 📌 **Source:** [Pittsburgh 311 Data](https://data.wprdc.org/dataset/pittsburgh-311-data) and [311 Data Archive](https://data.wprdc.org/dataset/311-data) — Western Pennsylvania Regional Data Center (WPRDC)**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📐 CONFIDENCE SCORING METHODOLOGY
# ============================================================

## How We Calculate Confidence

The Data Concierge uses a **weighted composite score** to assess the reliability
of each answer. The final confidence score is a weighted average of five independent
factors, each measuring a different aspect of answer quality.

### Scoring Formula

```
Final Score = (0.25 × Query Interpretation)
            + (0.25 × Source Authority)
            + (0.20 × Retrieval Match)
            + (0.15 × Data Recency)
            + (0.15 × Computation Reliability)
```

### Factor Descriptions

| Factor | Weight | What It Measures | How It's Calculated |
|--------|--------|------------------|---------------------|
| **Query Interpretation** | 25% | How well the system understood the query | Entity extraction confidence × intent classification confidence |
| **Source Authority** | 25% | Trustworthiness of the data source | Pre-assigned per source (BLS/Census: 0.95, Data Commons: 0.90, CKAN: 0.85) |
| **Retrieval Match** | 20% | How well the retrieved data matches the query | Retrieval score, boosted by observation count (up to 5 observations) |
| **Data Recency** | 15% | How fresh the data is | 1.0 if within expected update cycle, decays to 0.4 floor for older data |
| **Computation Reliability** | 15% | Accuracy of the computation method | By type: direct lookup 1.0, trend analysis 0.85, statistical inference 0.70 |

### Confidence Levels

| Level | Score Range | Interpretation |
|-------|-------------|----------------|
| 🟢 **HIGH** | ≥ 85% | Results are reliable and well-supported by authoritative data |
| 🟡 **MEDIUM** | 50% – 84% | Results are reasonable but may benefit from verification |
| 🔴 **LOW** | 25% – 49% | Results should be treated with caution; data may be incomplete |
| ⚫ **VERY LOW** | < 25% | Insufficient data; consider alternative sources or queries |

### Source Authority Ratings

| Data Source | Authority Score | Rationale |
|-------------|----------------|-----------|
| Bureau of Labor Statistics (BLS) | 0.95 | Official federal statistics, rigorous methodology |
| U.S. Census Bureau | 0.95 | Comprehensive national data collection |
| Bureau of Economic Analysis (BEA) | 0.95 | Official GDP and economic accounts |
| FRED (Federal Reserve) | 0.95 | Curated economic data from the Fed |
| Google Data Commons | 0.90 | Aggregated from authoritative sources |
| WPRDC (Pittsburgh) | 0.88 | Curated regional open data portal |
| Generic CKAN Portals | 0.85 | Quality varies by portal and dataset |

### Data Recency Decay

The recency score decays based on how old the data is relative to its expected
update frequency:

- **Within 1× update cycle**: 1.0 (fully current)
- **Within 2× update cycle**: 0.8
- **Within 4× update cycle**: 0.6
- **Older than 4× update cycle**: 0.4 (floor)

### Escalation Policy

When the final confidence score falls **below 50%** after **2 retrieval attempts**,
the system flags the query for human review rather than providing a potentially
unreliable answer.

---


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-25

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-25 13:16:05
- **Query**: Which city departments receive the most complaints?
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
